# Compare a ClearML HPO sweep
Reads each child task's HDF5 metrics, ranks by `train/level2/loss/0/0/0` at the *max common iteration* across all surviving tasks (= `min(last_iter_per_task)`), and overlays the curves.

**Why `train/level2/loss` and not `eval/...`:** `eval/*` and `eval_accumulated/*` are only logged in the post-training eval phase (`app.runApp` after training finishes). Tasks that hit NaN or got slurm-killed never reach that phase. The level-2 loss is computed at every training step inside `meta_step`, so it's the densest test-side signal you have.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from clearml import Task
from clearml.backend_api.session.client import APIClient

HPO_ID = "3c5b6798bcde46e48c50af7511727fca"
LOG_DIR = Path("/scratch/wlp9800/offline_logs")
METRIC = "train/level2/loss/0/0/0"
ABORT_SIGNAL = "train/level0/loss/0/0/0"  # app.py aborts on non-finite level0/loss
HP_KEY = "config/hyperparameters/meta1_sgd1_lr/value"  # the swept hyperparameter

## 1. Find child tasks
Query ClearML for tasks whose parent is the HPO task, then for each pull the swept hp value and check that its HDF5 file exists.

In [6]:
def find_h5(task_id: str) -> Path | None:
    matches = sorted(LOG_DIR.glob(f"metrics_{task_id}*.h5"))
    return matches[-1] if matches else None

def task_had_abort(task_id: str) -> bool:
    t = Task.get_task(task_id=task_id)
    out = t.get_reported_console_output(number_of_reports=500) or []
    return any("[abort]" in line for chunk in out for line in chunk.splitlines())

client = APIClient()
api_tasks = client.tasks.get_all(
    parent=HPO_ID,
    page_size=500,
    only_fields=["id", "name", "status"],
)
print(f"Found {len(api_tasks)} child tasks under HPO {HPO_ID}")

rows = []
for at in api_tasks:
    t = Task.get_task(task_id=at.id)
    params = t.get_parameters()
    raw_lr = params.get(HP_KEY)
    try:
        lr = float(raw_lr)
    except (TypeError, ValueError):
        lr = None
    h5 = find_h5(at.id)
    aborted = task_had_abort(at.id)
    rows.append({
        "task_id": at.id,
        "name": at.name,
        "status": at.status,
        "lr": lr,
        "h5": h5,
        "aborted": aborted,
    })

for r in sorted(rows, key=lambda r: (r["lr"] is None, r["lr"] or 0.0)):
    print(f"  {r['task_id'][:8]}  lr={r['lr']!r:14}  status={r['status']:10}  h5={'YES' if r['h5'] else 'MISSING':7}  abort={r['aborted']}")

Found 10 child tasks under HPO 83cbf18ad57e4d9588253a51b9702dbf
  9a410c61  lr=0.0001          status=stopped     h5=YES      abort=False
  bbfad6b5  lr=0.0001291549665  status=stopped     h5=YES      abort=False
  8b925aca  lr=0.00016681005372  status=stopped     h5=YES      abort=False
  c1f01cd3  lr=0.000215443469  status=stopped     h5=YES      abort=False
  b8660973  lr=0.00027825594022  status=stopped     h5=YES      abort=False
  01ed20b8  lr=0.00035938136638  status=completed   h5=YES      abort=True
  c94e416f  lr=0.00046415888336  status=completed   h5=YES      abort=True
  bc1d59fa  lr=0.00059948425032  status=completed   h5=YES      abort=True
  2d3dbd8d  lr=0.00077426368268  status=completed   h5=YES      abort=True
  215ec1a2  lr=0.001           status=completed   h5=YES      abort=True


## 2. Load the metric series per task
Drop tasks where the console contains `[abort]` — they hit a NaN (the HDF5 logger silently filters NaN via `np.nanmean`, so the recorded values look finite even when the task NaN-aborted). Also drop tasks with no HDF5 or no metric data.

In [7]:
series = {}  # task_id -> (lr, iters, vals)
dropped = []
for r in rows:
    if r["aborted"]:
        dropped.append((r, "NaN-aborted (console [abort])"))
        continue
    if r["h5"] is None or r["lr"] is None:
        dropped.append((r, "no h5 or no lr"))
        continue
    with h5py.File(r["h5"], "r") as f:
        if METRIC not in f or f"{METRIC}_iterations" not in f:
            dropped.append((r, "metric not logged"))
            continue
        vals = f[METRIC][:]
        iters = f[f"{METRIC}_iterations"][:]
    filled = iters >= 0
    if not filled.any():
        dropped.append((r, "no filled iters"))
        continue
    mask = filled & np.isfinite(vals)
    series[r["task_id"]] = (r["lr"], iters[mask], vals[mask])

print(f"Surviving tasks: {len(series)}")
for tid, (lr, its, _) in sorted(series.items(), key=lambda kv: kv[1][0]):
    print(f"  {tid[:8]}  lr={lr:.4e}  iters: {its[0]}..{its[-1]}  ({len(its)} points)")

print(f"\nDropped ({len(dropped)}):")
for r, reason in sorted(dropped, key=lambda x: (x[0]['lr'] is None, x[0]['lr'] or 0.0)):
    print(f"  {r['task_id'][:8]}  lr={r['lr']!r}  reason={reason}")

Surviving tasks: 5
  9a410c61  lr=1.0000e-04  iters: 1..62750  (62750 points)
  bbfad6b5  lr=1.2915e-04  iters: 1..68500  (68500 points)
  8b925aca  lr=1.6681e-04  iters: 1..63000  (63000 points)
  c1f01cd3  lr=2.1544e-04  iters: 1..66000  (66000 points)
  b8660973  lr=2.7826e-04  iters: 1..70000  (70000 points)

Dropped (5):
  01ed20b8  lr=0.00035938136638  reason=NaN-aborted (console [abort])
  c94e416f  lr=0.00046415888336  reason=NaN-aborted (console [abort])
  bc1d59fa  lr=0.00059948425032  reason=NaN-aborted (console [abort])
  2d3dbd8d  lr=0.00077426368268  reason=NaN-aborted (console [abort])
  215ec1a2  lr=0.001  reason=NaN-aborted (console [abort])


## 3. Rank at the max common iteration
`max_common_iter = min(last_iter)` across all surviving tasks — the largest iteration every task reached. For each task, take its loss at the closest iter ≤ `max_common_iter`. Also report each task's own-final loss (at its own last iter) as a sanity check.

In [8]:
max_common_iter = min(its[-1] for _, its, _ in series.values())
print(f"max common iteration = {max_common_iter}")

table = []
for tid, (lr, its, vals) in series.items():
    j = int(np.searchsorted(its, max_common_iter, side="right")) - 1
    common_loss = float(vals[j])
    common_iter_used = int(its[j])
    own_final_loss = float(vals[-1])
    own_final_iter = int(its[-1])
    table.append((lr, tid, common_iter_used, common_loss, own_final_iter, own_final_loss))

table.sort(key=lambda r: r[3])
print(f"\n{'lr':>12}  {'task':>10}  {'@common_iter':>14}  {'loss':>12}  {'@own_iter':>11}  {'own_loss':>12}")
for lr, tid, ci, cl, oi, ol in table:
    print(f"  {lr:>10.4e}  {tid[:8]:>10}  {ci:>14}  {cl:>12.6f}  {oi:>11}  {ol:>12.6f}")

best_lr = table[0][0]
print(f"\nBest lr by common-iter loss: {best_lr:.4e}")

max common iteration = 62750

          lr        task    @common_iter          loss    @own_iter      own_loss
  1.6681e-04    8b925aca           62750    327.558136        63000    325.616699
  1.2915e-04    bbfad6b5           62750    330.860962        68500    322.328522
  1.0000e-04    9a410c61           62750    330.968323        62750    330.968323
  2.1544e-04    c1f01cd3           62750    332.530731        66000    335.321045
  2.7826e-04    b8660973           62750    332.865906        70000    327.373138

Best lr by common-iter loss: 1.6681e-04


## 4. Overlay curves
Vertical dashed line marks `max_common_iter`. Legend is sorted by lr.

In [ ]:
SMOOTH_WINDOW = 1000  # rolling-mean window in iterations; tune to taste

def rolling_mean(x: np.ndarray, w: int) -> np.ndarray:
    if w <= 1 or x.size < w:
        return x
    c = np.cumsum(np.insert(x, 0, 0.0))
    return (c[w:] - c[:-w]) / w

fig, ax = plt.subplots(figsize=(12, 7))
items = sorted(series.items(), key=lambda kv: kv[1][0])
cmap = plt.get_cmap("viridis")
for i, (tid, (lr, its, vals)) in enumerate(items):
    color = cmap(i / max(len(items) - 1, 1))
    ax.plot(its, vals, linewidth=0.5, color=color, alpha=0.15)
    if vals.size >= SMOOTH_WINDOW:
        sm = rolling_mean(vals, SMOOTH_WINDOW)
        sm_its = its[SMOOTH_WINDOW - 1:]
        ax.plot(sm_its, sm, linewidth=1.8, color=color, label=f"lr={lr:.2e}")
    else:
        ax.plot(its, vals, linewidth=1.8, color=color, label=f"lr={lr:.2e}")
ax.axvline(max_common_iter, color="k", linestyle="--", alpha=0.4, label=f"common iter = {max_common_iter}")
ax.set_xlabel("iteration")
ax.set_ylabel(f"{METRIC} (rolling mean w={SMOOTH_WINDOW})")
ax.set_yscale("log")
ax.set_title(f"inner-lr sweep — HPO {HPO_ID[:8]}")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()